# 6.4 — Distributed Matrix Types

**Chapter 6, section 6.2.2**, and the starting point for **Exercise 4**.

**The question this notebook answers:** the four distributed matrix types are four *storage
layouts* for the same mathematical object, and the choice between them is made once, at
construction, because moving between them is a shuffle.

The same small matrix is built all four ways here, its representation counted each time, and the
operation each layout exists for is run: the Gramian on a `RowMatrix`, a join on an
`IndexedRowMatrix`, a sparse entry list on a `CoordinateMatrix`, and a matrix product on a
`BlockMatrix`.

**Which namespace.** All four types live in `pyspark.mllib.linalg.distributed` and have **no
DataFrame-based equivalent**. This notebook is therefore a deliberate excursion into the
RDD-based API, which is in maintenance mode but not deprecated. Vectors here come from
`pyspark.mllib.linalg`, not `pyspark.ml.linalg` — notebook 6.2 shows what happens when the two
are mixed.

Runs on a laptop in under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.4")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 0. One matrix, held in one place

Everything below is a representation of this $6\times3$ matrix. It is small enough to print, and
only eight of its eighteen entries are non-zero, which is what makes the record-counting in
section 5 worth doing.

In [2]:
X = np.array([[1., 0., 2.],
              [0., 3., 0.],
              [4., 0., 0.],
              [0., 0., 5.],
              [6., 7., 0.],
              [0., 0., 8.]])

print(X)
print("shape:", X.shape, " entries:", X.size, " non-zeros:", int((X != 0).sum()))

[[1. 0. 2.]
 [0. 3. 0.]
 [4. 0. 0.]
 [0. 0. 5.]
 [6. 7. 0.]
 [0. 0. 8.]]
shape: (6, 3)  entries: 18  non-zeros: 8


A distributed matrix has **long-typed** row and column indices and double values, stored in one or
more RDDs. The long indices are the first signal of intent: these are types for matrices whose
dimensions exceed what a 32-bit index can address, which is to say matrices that could never be
local. A six-row example is a scale model, and the operations are the real ones.

One constraint before any of them: the RDDs backing a distributed matrix **must be
deterministic**, because the matrix caches its size and a non-deterministic backing RDD leads to
errors.

---

## 1. `RowMatrix` — rows, without identity

The basic type: row-oriented, backed by an RDD of local vectors, **without meaningful row
indices**. The canonical case is a collection of feature vectors where the order of the rows
carries no information.

Its stated assumption repays attention: the number of columns is taken to be small enough that a
single row can reasonably be communicated to the driver and operated on by one node. A
`RowMatrix` may be extraordinarily **tall**; it must not be extraordinarily **wide**.

In [3]:
from pyspark.mllib.linalg import Vectors, Matrices
from pyspark.mllib.linalg.distributed import RowMatrix

rows = sc.parallelize([Vectors.dense(r) for r in X])
row_mat = RowMatrix(rows)

print("numRows():", row_mat.numRows())
print("numCols():", row_mat.numCols())
print("records in the backing RDD:", row_mat.rows.count())

numRows(): 6
numCols(): 3


records in the backing RDD: 6


The constructor takes the RDD directly and the dimensions need not be supplied: `numRows()` and
`numCols()` compute them from the data on first request, which is one reason the backing RDD must
be deterministic.

The wrapper is thin by design, and its value is in the operations it carries. The one to remember
is `computeGramianMatrix()`, which computes $X^\top X$ — a $d \times d$ matrix, small whenever the
column count $d$ is modest, and computable distributively.

In [4]:
gramian = row_mat.computeGramianMatrix().toArray()
print("distributed Gramian X^T X:\n", gramian)
print("\nNumPy, in one process:\n", X.T @ X)
assert np.allclose(gramian, X.T @ X)

distributed Gramian X^T X:
 [[53. 42.  2.]
 [42. 58.  0.]
 [ 2.  0. 93.]]

NumPy, in one process:
 [[53. 42.  2.]
 [42. 58.  0.]
 [ 2.  0. 93.]]


### Why the Gramian is the operation to remember

The closed-form solution of linear regression is

$$\hat{\beta} = (X^\top X)^{-1} X^\top y ,$$

and every distributed part of it is already in hand. $X^\top X$ is the Gramian, computed above
over a matrix that never had to fit on one machine. $X^\top y$ is a `map` and a `reduce` — the
pattern of notebook 6.1, one $d$-element vector per partition. Both results are of size $d$, and
the inversion and the solve happen on the driver, on objects that are $d \times d$.

A billion rows change nothing about the last step.

In [5]:
beta_true = np.array([1.5, -2.0, 0.5])
y = X @ beta_true                      # a noiseless target, so the fit must recover beta exactly

# X^T y, distributed: the map/reduce pattern of notebook 6.1
labelled = sc.parallelize(list(zip(X, y)))
xty = labelled.map(lambda r: r[0] * r[1]).reduce(lambda a, b: a + b)

beta_hat = np.linalg.solve(gramian, xty)

print("X^T y      :", xty)
print("beta_hat   :", beta_hat)
print("beta_true  :", beta_true)
assert np.allclose(beta_hat, beta_true)

X^T y      : [ -3.5 -53.   49.5]
beta_hat   : [ 1.5 -2.   0.5]
beta_true  : [ 1.5 -2.   0.5]


The remaining `RowMatrix` operations are the standard tools for a wide feature matrix:
`computeSVD(k)`, `computePrincipalComponents(k)`, `columnSimilarities()`,
`computeColumnSummaryStatistics()` and `tallSkinnyQR()`.

The Gramian, covariance and principal-component computations share a documented limit of
**65,535 columns**, which is the "columns not huge" assumption made precise.

In [6]:
stats = row_mat.computeColumnSummaryStatistics()
print("column means :", stats.mean())
print("non-zeros    :", stats.numNonzeros())
print("count        :", stats.count())

column means : [1.83333333 1.66666667 2.5       ]
non-zeros    : [3. 2. 3.]
count        : 6


---

## 2. `IndexedRowMatrix` — rows that can be joined

A `RowMatrix` with meaningful row indices, backed by an RDD of `IndexedRow` objects, each a
wrapper over a `(long, vector)` pair. The indices are what allow rows to be **identified and
joined**.

In [7]:
from pyspark.mllib.linalg.distributed import IndexedRow, IndexedRowMatrix

# explicitly, with the IndexedRow class ...
indexed_rows = sc.parallelize([IndexedRow(i, Vectors.dense(r)) for i, r in enumerate(X)])
# ... or equivalently from (long, vector) tuples
indexed_rows_tuples = sc.parallelize([(i, Vectors.dense(r)) for i, r in enumerate(X)])

indexed_mat = IndexedRowMatrix(indexed_rows)
print("numRows():", indexed_mat.numRows(), " numCols():", indexed_mat.numCols())
for r in sorted(indexed_mat.rows.collect(), key=lambda r: r.index)[:3]:
    print(" index", r.index, "->", r.vector)

numRows(): 6  numCols(): 3


 index 0 -> [1.0,0.0,2.0]
 index 1 -> [0.0,3.0,0.0]
 index 2 -> [4.0,0.0,0.0]


The join case is the one to remember. Two datasets keyed by the same identifier — a user, a
document, a sensor — can be aligned row by row only if the rows **carry** the identifier, and a
bare `RowMatrix` has already thrown it away.

In [8]:
# a second matrix keyed the same way: one extra measurement per sensor
extra = sc.parallelize([(i, Vectors.dense([float(10 + i)])) for i in range(6)])

joined = (indexed_mat.rows
          .map(lambda r: (r.index, r.vector))
          .join(extra)
          .map(lambda kv: IndexedRow(kv[0], Vectors.dense(np.concatenate(
              [kv[1][0].toArray(), kv[1][1].toArray()]))))
          )

widened = IndexedRowMatrix(joined)
print("joined shape:", widened.numRows(), "x", widened.numCols())
for r in sorted(widened.rows.collect(), key=lambda r: r.index):
    print(" index", r.index, "->", r.vector)

joined shape: 6 x 4
 index 0 -> [1.0,0.0,2.0,10.0]
 index 1 -> [0.0,3.0,0.0,11.0]
 index 2 -> [4.0,0.0,0.0,12.0]
 index 3 -> [0.0,0.0,5.0,13.0]
 index 4 -> [6.0,7.0,0.0,14.0]
 index 5 -> [0.0,0.0,8.0,15.0]


Dropping the indices again is the **cheap** direction: `toRowMatrix()` moves no data, because each
row stays where it is. Reattaching indices to an unindexed matrix would require deciding which row
is which, and there is no operation for it — the information, once discarded, is gone.

In [9]:
back_to_rows = indexed_mat.toRowMatrix()
print("type:", type(back_to_rows).__name__)
print("rows:", back_to_rows.rows.count(), " -- the indices are simply not there any more")
print(back_to_rows.rows.first())

type: RowMatrix
rows: 6  -- the indices are simply not there any more
[1.0,0.0,2.0]


---

## 3. `CoordinateMatrix` — only the non-zeros

Coordinate list (COO) format: an RDD of `MatrixEntry` objects, each wrapping `(i, j, value)`. The
documentation is unusually direct about when to use it — **only when both dimensions are huge and
the matrix is very sparse**.

This is the distributed rendering of notebook 6.3's argument. A user–item rating matrix with
millions of users and millions of items cannot be stored by rows, because each row would be a
millions-long vector violating `RowMatrix`'s width assumption. Its non-zeros, the ratings that
actually exist, form a modest list of triples.

In [10]:
from pyspark.mllib.linalg.distributed import CoordinateMatrix, MatrixEntry

entries = sc.parallelize([MatrixEntry(i, j, X[i, j])
                          for i in range(X.shape[0]) for j in range(X.shape[1])
                          if X[i, j] != 0])

coord_mat = CoordinateMatrix(entries)
print("numRows():", coord_mat.numRows(), " numCols():", coord_mat.numCols())
print("records in the backing RDD:", coord_mat.entries.count(), "-- the non-zeros only")
for e in sorted(coord_mat.entries.take(4), key=lambda e: (e.i, e.j)):
    print(f"  ({e.i}, {e.j}) = {e.value}")

numRows(): 6  numCols(): 3


records in the backing RDD: 8 -- the non-zeros only
  (0, 0) = 1.0
  (0, 2) = 2.0
  (1, 1) = 3.0
  (2, 0) = 4.0


### Where the global-shuffle warning becomes concrete

To build **rows** out of entries, every entry with row index $i$ must arrive at the same place.
That is a shuffle keyed by row, and the same holds block-wise for `toBlockMatrix()`. The
conversions exist and are correct, and each one is a full repartitioning of the data by a new key.

The lineage says so out loud.

In [11]:
converted = coord_mat.toRowMatrix()
lineage = converted.rows.toDebugString().decode()
print(lineage)
print("\nShuffledRDD appears in the lineage:", "ShuffledRDD" in lineage)

(4) MapPartitionsRDD[98] at mapPartitions at PythonMLLibAPI.scala:1345 []
 |  MapPartitionsRDD[97] at map at IndexedRowMatrix.scala:90 []
 |  MapPartitionsRDD[96] at map at CoordinateMatrix.scala:89 []
 |  ShuffledRDD[95] at groupByKey at CoordinateMatrix.scala:87 []
 +-(4) MapPartitionsRDD[94] at map at CoordinateMatrix.scala:87 []
    |  MapPartitionsRDD[79] at map at PythonMLLibAPI.scala:1171 []
    |  MapPartitionsRDD[78] at rdd at PythonMLLibAPI.scala:1171 []
    |  SQLExecutionRDD[77] at rdd at PythonMLLibAPI.scala:1171 []
    |  MapPartitionsRDD[76] at rdd at PythonMLLibAPI.scala:1171 []
    |  MapPartitionsRDD[75] at rdd at PythonMLLibAPI.scala:1171 []
    |  MapPartitionsRDD[74] at applySchemaToPythonRDD at DirectMethodHandleAccessor.java:103 []
    |  MapPartitionsRDD[73] at map at SerDeUtil.scala:71 []
    |  MapPartitionsRDD[72] at mapPartitions at SerDeUtil.scala:119 []
    |  PythonRDD[71] at RDD at PythonRDD.scala:59 []
    |  ParallelCollectionRDD[69] at readRDDFromFile

In [12]:
# the same values arrive, whichever way round they were stored
as_dense = np.zeros_like(X)
for e in coord_mat.entries.collect():
    as_dense[e.i, e.j] = e.value
assert np.allclose(as_dense, X)
print("CoordinateMatrix reconstructs X exactly:", np.allclose(as_dense, X))

CoordinateMatrix reconstructs X exactly: True


A pipeline that constructs a `CoordinateMatrix`, converts it to rows, and later converts it back
has paid for two shuffles that a better initial choice might have avoided. That is the whole
reason the chapter frames this subsection as a design problem rather than a catalogue.

---

## 4. `BlockMatrix` — the only one that multiplies

Backed by an RDD of `((blockRowIndex, blockColIndex), sub-matrix)` pairs: a pair of block
coordinates together with a local matrix of size `rowsPerBlock` × `colsPerBlock` at that position.

It is the **only** one of the four types that supports matrix–matrix operations, `add` and
`multiply`. The reason block storage is what unlocks multiplication is one sentence of linear
algebra: a matrix product decomposes into products of aligned sub-blocks, each of which is a
local multiplication of two ordinary matrices. The block layout turns a distributed multiply into
a shuffle that pairs the right blocks, followed by purely local computation.

In [13]:
from pyspark.mllib.linalg.distributed import BlockMatrix

# the documentation's own example: two 3x2 blocks stacked vertically into a 6x2 matrix
blocks = sc.parallelize([((0, 0), Matrices.dense(3, 2, [1, 2, 3, 4, 5, 6])),
                         ((1, 0), Matrices.dense(3, 2, [7, 8, 9, 10, 11, 12]))])

block_mat = BlockMatrix(blocks, 3, 2)
print("numRows():", block_mat.numRows(), " numCols():", block_mat.numCols())
print("blocks in the backing RDD:", block_mat.blocks.count())
block_mat.validate()
print("validate() passed")
print("\nas one local matrix:\n", block_mat.toLocalMatrix().toArray())

numRows(): 6  numCols(): 2


blocks in the backing RDD: 2
validate() passed

as one local matrix:
 [[ 1.  4.]
 [ 2.  5.]
 [ 3.  6.]
 [ 7. 10.]
 [ 8. 11.]
 [ 9. 12.]]


Note the values: `Matrices.dense(3, 2, [1, 2, 3, 4, 5, 6])` is **column-major**, so that block is
$\begin{pmatrix}1&4\\2&5\\3&6\end{pmatrix}$, not the row-major reading. Notebook 6.3 is where that
convention is established, and here is where forgetting it would go unnoticed.

Now the operation the type exists for.

In [14]:
B = BlockMatrix(sc.parallelize([((0, 0), Matrices.dense(2, 2, [1, 3, 2, 4]))]), 2, 2)

product = block_mat.multiply(B)
print("A (6x2) x B (2x2) = ", product.numRows(), "x", product.numCols())
print(product.toLocalMatrix().toArray())

A_np = block_mat.toLocalMatrix().toArray()
B_np = B.toLocalMatrix().toArray()
assert np.allclose(product.toLocalMatrix().toArray(), A_np @ B_np)
print("\nagrees with NumPy's A @ B:", np.allclose(product.toLocalMatrix().toArray(), A_np @ B_np))

A (6x2) x B (2x2) =  6 x 2
[[13. 18.]
 [17. 24.]
 [21. 30.]
 [37. 54.]
 [41. 60.]
 [45. 66.]]

agrees with NumPy's A @ B: True


`toLocalMatrix()` is this section's equivalent of `collect()`: it materializes the whole
distributed matrix on the driver, and it is safe exactly when the matrix is small enough that
distributing it was unnecessary. It is used above to check the answer, which is the one honest
reason to call it.

---

## 5. The same matrix, four ways, counted

What each layout actually stores, for the one $6\times3$ matrix defined at the top.

In [15]:
summary = [
    ("RowMatrix",        row_mat.rows.count(),        "local vectors, one per row"),
    ("IndexedRowMatrix", indexed_mat.rows.count(),    "(long, vector) pairs"),
    ("CoordinateMatrix", coord_mat.entries.count(),   "(i, j, value) triples -- non-zeros only"),
    ("BlockMatrix",      block_mat.blocks.count(),    "((i, j), sub-matrix) pairs"),
]
print(f"{'type':<19}{'records':>9}   what a record is")
for name, n, what in summary:
    print(f"{name:<19}{n:>9}   {what}")

print(f"\nX has {X.size} entries, {int((X != 0).sum())} of them non-zero.")
print("Only CoordinateMatrix's record count tracks the non-zeros; the row-oriented")
print("types pay for every entry, and BlockMatrix pays per block whatever it contains.")

type                 records   what a record is
RowMatrix                  6   local vectors, one per row
IndexedRowMatrix           6   (long, vector) pairs
CoordinateMatrix           8   (i, j, value) triples -- non-zeros only
BlockMatrix                2   ((i, j), sub-matrix) pairs

X has 18 entries, 8 of them non-zero.
Only CoordinateMatrix's record count tracks the non-zeros; the row-oriented
types pay for every entry, and BlockMatrix pays per block whatever it contains.


---

## 6. The decision procedure

The four types condense into four questions, asked in this order:

| If … | use | because |
|---|---|---|
| the rows are feature vectors and the width is modest | `RowMatrix` | rows distribute; one row must fit on one node |
| rows must be identified or joined | `IndexedRowMatrix` | the index is the join key, and `RowMatrix` has discarded it |
| both dimensions are large and the matrix is very sparse | `CoordinateMatrix` | a row would be too wide to hold; the non-zeros are a short list |
| a matrix–matrix product is needed | `BlockMatrix` | it is the only type with `add` and `multiply` |

**Choose once, at construction, with the eventual operation in mind, because moving between the
choices is a shuffle.** A conversion inside a training loop is paid once per iteration.

### One performance note

MLlib's linear algebra runs, on the JVM side, through `dev.ludovic.netlib` with Breeze. BLAS is
the standard these call into, and its optimized native implementations include Intel MKL and
OpenBLAS. When a native BLAS is installed, `dev.ludovic.netlib` calls it; when none is found it
falls back to a pure-JVM implementation and prints a warning of the form

```
Failed to load implementation from: dev.ludovic.netlib.blas.JNIBLAS
```

The official Spark binaries **do not bundle a native BLAS**, so the fallback is common on
unconfigured clusters. The practical consequence is a diagnostic habit: before concluding that
MLlib's linear algebra is slow, check the executor logs for that warning, because the difference
between native and pure-JVM BLAS is often the whole difference being measured.

---

## Conclusion

Four layouts, one mathematical object, and the layout is a running cost rather than a formality.
`RowMatrix` distributes whole rows and gives up identity for it; `IndexedRowMatrix` keeps the
identity and can therefore join; `CoordinateMatrix` distributes individual non-zeros and is the
only one whose storage tracks sparsity; `BlockMatrix` distributes sub-matrices and is the only one
that can multiply.

The Gramian is the operation that connects this notebook to the rest of the course: $X^\top X$ is
$d \times d$ regardless of how many rows $X$ has, so the closed-form regression solution is
computable at any scale once the Gramian and $X^\top y$ are in hand — and both of those are a
per-partition computation plus an associative combine, which is notebook 6.5's subject.

Exercise 4 asks the decision-procedure question of four workloads, and Exercise 5 the namespace
question that notebook 6.2 demonstrates.

In [16]:
spark.stop()